In [ ]:
import os,hashlib
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import numpy as np
import pandas as pd
import streamlit as st
import tensorflow as tf
import tensorflow_hub as hub

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

HUB_URL = "https://tfhub.dev/google/nnlm-en-dim128/2"
MAX_TOKENS = 12000
SEQUENCE_LENGTH = 60
EMBEDDING_DIM = 128
CSV_PATH = os.path.join(
    os.path.dirname(os.path.abspath(__file__)),
    "Dataset.csv"
)

st.set_page_config(
    page_title="Text Classification Studio",
    layout="wide"
)

st.title("Text Classification Studio")
st.header("Dataset")

uploaded_file = st.file_uploader(
    "Upload your Csv dataset",
    type=["csv"]
)

def load_data(file):
    if file is not None:
        df = pd.read_csv(file)

    else:
        if not os.path.exists(CSV_PATH):
            raise FileNotFoundError(
                "Dataset Not Found. Upload your Csv"
            )

        df = pd.read_csv(CSV_PATH)

    required_columns = {"statement", "status"}

    if not required_columns.issubset(df.columns):
        raise ValueError(
            "Csv must Contain Statement & Status Columns"
        )

    df = df[["statement", "status"]].copy()

    df["statement"] = df["statement"].astype("string").str.strip()
    df["status"] = pd.to_numeric(df["status"], errors="coerce")

    df = df.dropna(subset=["statement", "status"])
    df = df[df["statement"] != ""]
    df = df.drop_duplicates(subset=["statement"])

    if not df["status"].isin([0, 1]).all():
        raise ValueError("Status must Contain only 0 & 1")

    if df["status"].nunique() != 2:
        raise ValueError("Both Sentiment Classes must be Present")

    if len(df) < 10:
        raise ValueError("Dataset must Contain at least 10 Valid Rows")

    df["status"] = df["status"].astype("float32")
    return df

try:
    df = load_data(uploaded_file)

except Exception as error:
    st.error(str(error))
    st.stop()

col1, col2, col3 = st.columns(3)

col1.metric("Total Samples = ", len(df))
col2.metric("Positive Samples = ", int((df["status"] == 1).sum()))
col3.metric("Negative Samples = ", int((df["status"] == 0).sum()))

with st.expander("View Dataset"):
    st.dataframe(df.head(20), use_container_width=True)

dataset_hash = hashlib.sha256(
    pd.util.hash_pandas_object(df, index=True).values.tobytes()
).hexdigest()


def split_data(df):
    train_df, temp_df = train_test_split(
        df,
        test_size=0.20,
        random_state=42,
        stratify=df["status"]
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=42,
        stratify=temp_df["status"]
    )

    return (
        train_df["statement"].to_numpy(dtype=str),
        train_df["status"].to_numpy(dtype=np.float32),

        val_df["statement"].to_numpy(dtype=str),
        val_df["status"].to_numpy(dtype=np.float32),

        test_df["statement"].to_numpy(dtype=str),
        test_df["status"].to_numpy(dtype=np.float32)
    )


def build_vectorizer(x_train):
    vectorizer = tf.keras.layers.TextVectorization(
        max_tokens=MAX_TOKENS,
        output_mode="int",
        output_sequence_length=SEQUENCE_LENGTH,
        standardize="lower_and_strip_punctuation"
    )

    vectorizer.adapt(x_train)
    return vectorizer

class Network(tf.keras.Model):

    def __init__(self, vectorizer, network_type):
        super().__init__()

        self.vectorizer = vectorizer

        self.embedding = tf.keras.layers.Embedding(
            input_dim=MAX_TOKENS,
            output_dim=EMBEDDING_DIM,
            mask_zero=True
        )

        if network_type == "SimpleRNN":
            self.recurrent = tf.keras.layers.SimpleRNN(64)

        elif network_type == "LSTM":
            self.recurrent = tf.keras.layers.LSTM(64)

        elif network_type == "GRU":
            self.recurrent = tf.keras.layers.GRU(64)

        else:
            raise ValueError("Invalid Network Architecture")

        self.dropout = tf.keras.layers.Dropout(0.35)
        self.dense = tf.keras.layers.Dense(
            32,
            activation="relu"
        )
        self.output_layer = tf.keras.layers.Dense(
            1,
            activation="sigmoid"
        )

    def call(self, inputs, training=False):
        x = self.vectorizer(inputs)
        x = self.embedding(x)
        x = self.recurrent(x)
        x = self.dropout(x, training=training)
        x = self.dense(x)

        return self.output_layer(x)

def build_online_model():
    embedding = hub.KerasLayer(
        HUB_URL,
        trainable=False,
        name="pretrained_text_embedding"
    )

    model = tf.keras.Sequential([
        tf.keras.Input(shape=(), dtype=tf.string),

        embedding,
        tf.keras.layers.Dense(
            128,
            activation="relu"
        ),
        tf.keras.layers.Dropout(0.35),
        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),
        tf.keras.layers.Dropout(0.25),
        tf.keras.layers.Dense(
            1,
            activation="sigmoid"
        )
    ])

    return model

def compile_model(model):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001
        ),

        loss="binary_crossentropy",

        metrics=[
            "accuracy",
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.AUC(name="auc")
        ]
    )

@st.cache_resource()
def train_and_cache_model(
    model_mode,
    network_type,
    current_dataset_hash,
    epochs,
    batch_size,
    _df
):
    tf.keras.utils.set_random_seed(42)
    x_train, y_train, x_val, y_val, x_test, y_test = split_data(_df)

    if model_mode == "Online Model":
        model = build_online_model()

    else:
        vectorizer = build_vectorizer(x_train)

        model = Network(
            vectorizer,
            network_type
        )

        model(np.array(["Model initialization"], dtype=str))

    compile_model(model)

    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )

    history = model.fit(
        x_train,
        y_train,

        validation_data=(x_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stopping],
        verbose=0
    )

    results = model.evaluate(
        x_test,
        y_test,
        batch_size=batch_size,
        return_dict=True,
        verbose=0
    )

    predictions = model.predict(
        x_test,
        batch_size=batch_size,
        verbose=0
    ).flatten()

    predicted_classes = (predictions >= 0.5).astype(int)

    report = classification_report(
        y_test.astype(int),
        predicted_classes,
        target_names=["Negative", "Positive"],
        output_dict=True,
        zero_division=0
    )

    matrix = confusion_matrix(
        y_test.astype(int),
        predicted_classes
    )

    return {
        "model": model,
        "history": history.history,
        "results": results,
        "report": report,
        "matrix": matrix
    }

def model_interface(model_mode, network_type=None):

    st.divider()
    st.subheader("Training Configuration")

    col1, col2 = st.columns(2)

    with col1:
        epochs = st.slider(
            "Training Epochs",
            min_value=1,
            max_value=30,
            value=10,
            key=f"{model_mode}_epochs"
        )

    with col2:
        batch_size = st.selectbox(
            "Batch Size",
            [16, 32, 64],
            index=1,
            key=f"{model_mode}_batch"
        )

    if model_mode == "Online Model":

        st.info(
            "Uses a Pretrained TensorFlow Hub Embedding, Requires Internet Connection"
        )
    else:
        st.info(
            f"Uses a Custom Trainable Embedding & {str(network_type).title()}"
        )

    st.subheader("Text Classification")
    statement = st.text_area(
        "Enter a Statement",
        height=130,
        key=f"{model_mode}_statement"
    )

    if st.button(
        "Predict",
        type="primary",
        use_container_width=True,
        key=f"{model_mode}_predict"
    ):

        if not statement.strip():
            st.warning("Please Enter some Text")
        else:

            try:

                trained = train_and_cache_model(
                    model_mode,
                    network_type,
                    dataset_hash,
                    epochs,
                    batch_size,
                    df
                )

                model = trained["model"]
                prediction = model.predict(
                    np.array([statement.strip()], dtype=str),
                    verbose=0
                )[0][0]
                sentiment = (
                    "Positive" if prediction >= 0.5 else "Negative"
                )
                confidence = (
                    prediction if prediction >= 0.5
                    else 1 - prediction
                )

                st.session_state[
                    f"{model_mode}_last_prediction"
                ] = {
                    "statement": statement,
                    "sentiment": sentiment,
                    "confidence": confidence
                }

                st.session_state[
                    f"{model_mode}_cache_key"
                ] = (
                    dataset_hash,
                    epochs,
                    batch_size,
                    network_type
                )

                st.success("Prediction completed!")

            except Exception as error:
                st.error(f"Model error: {error}")

    prediction_data = st.session_state.get(
        f"{model_mode}_last_prediction"
    )

    if prediction_data:

        st.divider()
        st.subheader("Prediction Result")

        st.write(f"Statement : {prediction_data['statement']}")

        if prediction_data["sentiment"] == "Positive":
            st.success("Sentiment : Positive")
        else:
            st.error("Sentiment : Negative")

    cache_key = (
        dataset_hash,
        epochs,
        batch_size,
        network_type
    )

    if st.session_state.get(
        f"{model_mode}_cache_key"
    ) == cache_key:

        trained = train_and_cache_model(
            model_mode,
            network_type,
            dataset_hash,
            epochs,
            batch_size,
            df
        )

        st.divider()
        st.subheader("Training Results")

        history_df = pd.DataFrame(trained["history"])

        st.write("Accuracy")
        st.line_chart(
            history_df[["accuracy", "val_accuracy"]]
        )

        st.write("Loss")
        st.line_chart(
            history_df[["loss", "val_loss"]]
        )

        st.divider()
        st.subheader("Test Evaluation")

        results = trained["results"]
        metric_cols = st.columns(5)

        metric_items = [
            ("Accuracy", "accuracy"),
            ("Precision", "precision"),
            ("Recall", "recall"),
            ("AUC", "auc"),
            ("Loss", "loss")
        ]

        for column, (label, key) in zip(
            metric_cols,
            metric_items
        ):
            column.metric(
                label,
                f"{results[key]:.4f}"
            )

        st.write("Classification Report")

        report_df = pd.DataFrame(
            trained["report"]
        ).transpose()

        st.dataframe(
            report_df.round(4),
            use_container_width=True
        )

        st.write("**Confusion Matrix**")

        matrix_df = pd.DataFrame(
            trained["matrix"],
            index=["Actual Negative", "Actual Positive"],
            columns=["Predicted Negative", "Predicted Positive"]
        )

        st.dataframe(
            matrix_df,
            use_container_width=True
        )

online_tab, offline_tab = st.tabs([
    "Online Model",
    "Offline Model"
])

with online_tab:

    st.header("Online Model")

    model_interface(
        model_mode="Online Model"
    )

with offline_tab:

    st.header("Offline Model")
    network_type = st.selectbox(
        "Choose Architecture",
        ["SimpleRNN", "LSTM", "GRU"],
        key="offline_architecture"
    )

    model_interface(
        model_mode="Offline Model",
        network_type=network_type
    )